# Summary
This is the master pipeline. It consists of work that has been done in previous notebooks.

- Section 1, Data Enrichment
- Section 2, Feature Engineering
- Section 3, Class Imbalance, Hardware Limitations and Conclusions

# Section 1, Data Enrichment

In [7]:
import utils
utils.import_data_science(globals())

In [8]:
metadata_dir = utils.check_os_get_fma_metadata_path()

os.listdir(metadata_dir) # should contain fma_metadata folder

['capitmn.mp3',
 'echonest.csv',
 'features.csv',
 'gang_starr.mp3',
 'genres.csv',
 'not_found.pickle',
 'raw_albums.csv',
 'raw_artists.csv',
 'raw_echonest.csv',
 'raw_genres.csv',
 'raw_tracks.csv',
 'README.txt',
 'records_ready.csv',
 'records_ready.parquet',
 'tfz.mp3',
 'tracks.csv']

In [9]:
# features = pd.read_csv(f"{metadata_dir}/features.csv", low_memory=False)

In [10]:
# reading genres
genres = pd.read_csv(f"{metadata_dir}/raw_genres.csv", low_memory=False)

# selecting Rock, Classical, Electronic, Hip-hop and Pop as per genre_parent_id
target_genres = genres[genres['genre_parent_id'].isin([12,10,5,21,15])]

# clearing for joining purposes
genres        = genres.rename(columns={"genre_id":"gid"})
target_genres = target_genres.rename(columns={"genre_parent_id":"gid"})

# casting
genres['gid']    = genres['gid'].astype('Int64')
target_genres['gid'] = target_genres['gid'].astype('Int64')

# fixing the id's
genres['gid'] = genres['gid'] - 1
target_genres['gid'] = target_genres['gid'] - 1

# necessary columns
columns_of_interest = ['genre_id', 'genre_handle', 'genre_title', 'genre_handle_parent', 'genre_title_parent']

# finished
genres_with_subgenres = target_genres.join(genres, on="gid", rsuffix="_parent")[columns_of_interest]

# Count of higher level genres for each subgenre
genres_with_subgenres['genre_title_parent'].value_counts()
# genres_with_subgenres

genre_title_parent
Rock          15
Electronic    14
Hip-Hop        7
Classical      7
Pop            2
Name: count, dtype: int64

## Tracks Metadata
### tracks.csv - data as is
- ```tracks[tracks['genre_top'].isin(['Rock', 'Electronic', 'Hip-Hop', 'Pop', 'Classical'])]``` -> 30668
- ```tracks["genre_top"].value_counts()``` -> Songs per genre provided by fma_dataset
    ```
    genre_top
    Rock          14182
    Electronic     9372
    Hip-Hop        3552
    Pop            2332
    Classical      1230
    Name: count, dtype: int64
    ```

- Percentage view:
```
genre_top
Rock          46.243642 %
Electronic    30.559541 %
Hip-Hop       11.582105 %
Pop            7.604017 %
Classical      4.010695 %
```
### enhancing the dataset with generalistaion of subgenres
- ```genres_with_subgenres['genre_title_parent'].value_counts()``` -> Additional subgenres transformed into genres
  ```
    genre_title_parent
    Rock          15
    Electronic    14
    Hip-Hop        7
    Classical      7
    Pop            2
    Name: count, dtype: int64
  ```

## Statistics

in top 5 genres: 
- 30668 ready songs / 106575 with genres in general

do not have top genre: 
- 56997 songs 


## Conclusion
We want to extract more genres from 56997 songs

In [11]:
tracks = pd.read_csv(f"{metadata_dir}/tracks.csv",
                     low_memory=False,
                    skiprows=1)

In [12]:
target_ready_tracks = tracks[tracks['genre_top'].isin(['Rock', 'Electronic', 'Hip-Hop', 'Pop', 'Classical'])]

print("percentage view")
target_ready_tracks['genre_top'].value_counts() / 30668 * 100


percentage view


genre_top
Rock          46.243642
Electronic    30.559541
Hip-Hop       11.582105
Pop            7.604017
Classical      4.010695
Name: count, dtype: float64

In [13]:
empty_genre_top = tracks[tracks['genre_top'].isnull() & tracks['genres'].notnull()]

In [14]:
empty_genre_top.iloc[1][['genre_top','genres']]

genre_top          NaN
genres       [76, 103]
Name: 6, dtype: object

In [15]:
genres_with_subgenres.iloc[:5]

,genre_id,genre_handle,genre_title,genre_handle_parent,genre_title_parent
22,25,Punk,Punk,Rock,Rock
23,26,Post-Rock,Post-Rock,Rock,Rock
24,27,Lo-fi,Lo-Fi,Rock,Rock
26,31,Metal,Metal,Rock,Rock
29,36,Krautrock,Krautrock,Rock,Rock


In [16]:
import re
def extract_subgenres(a: str):
    vals = []
    for i in re.findall(r'\d+', a):
        vals.append(int(i))
    return vals

### Preparing a logic for genre generalisation

In [17]:
genres_with_subgenres.set_index('genre_id')

value_counts = {
        'Pop': 0,
        'Rock': 0,
        'Electronic': 0,
        'Hip-Hop':0,
        'Classical':0
        }


def get_parent_genre(raw_genres):
    subgenres = extract_subgenres(raw_genres)
    
    for sub in subgenres:
        exists = genres_with_subgenres[genres_with_subgenres['genre_id'].eq(sub)]
        
        if not exists.empty:
            let = exists['genre_title_parent'].iloc[0]
            if let in value_counts.keys():
                value_counts[let] += 1
            return let
            break
            
        else:
            return None


## Applying

In [18]:
new_genres = empty_genre_top.copy()
new_genres['genre_top'] = empty_genre_top['genres_all'].apply(get_parent_genre)
print(value_counts)

{'Pop': 355, 'Rock': 4820, 'Electronic': 4870, 'Hip-Hop': 1272, 'Classical': 433}


In [19]:
new_genres[new_genres['genre_top'].notnull()]['genre_top'].value_counts()

# Counts of tracks with generalised genre from subgenres

genre_top
Electronic    4870
Rock          4820
Hip-Hop       1272
Classical      433
Pop            355
Name: count, dtype: int64

In [20]:
new_genres[new_genres['genre_top'].notnull()]['genre_top'].value_counts() / 56976 * 100 # procentowy zysk gatunków przez scalenie

# Percentage overall gain due to genre generalisation

genre_top
Electronic    8.547459
Rock          8.459702
Hip-Hop       2.232519
Classical     0.759969
Pop           0.623069
Name: count, dtype: float64

## Generalisation Gain
```new_genres[new_genres['genre_top'].notnull()]['genre_top'].value_counts() / 56976 * 100 ```
```
genre_top from 56976 songs without genre_top
Rock          8.270149 %
Electronic    2.283418 %
Pop           1.105729 %
Hip-Hop       0.270289 %
Classical     0.107063 %
Name: count, dtype: float64
```



In [21]:
df = pd.concat([target_ready_tracks, new_genres], ignore_index=True)

In [22]:
df['genre_top'].value_counts()

genre_top
Rock          19002
Electronic    14242
Hip-Hop        4824
Pop            2687
Classical      1663
Name: count, dtype: int64

In [23]:
new_genres[new_genres['genre_top'].notnull()]['genre_top'].value_counts()

genre_top
Electronic    4870
Rock          4820
Hip-Hop       1272
Classical      433
Pop            355
Name: count, dtype: int64

# Calculating gain in %

In [24]:
(new_genres[new_genres['genre_top'].notnull()]['genre_top'].value_counts() / df['genre_top'].value_counts() * 100).sort_values(ascending=False).astype(str) + "  %"

genre_top
Electronic     34.19463558488977  %
Hip-Hop       26.368159203980102  %
Classical      26.03728202044498  %
Rock           25.36575097358173  %
Pop           13.211760327502791  %
Name: count, dtype: object

In [25]:
columns_of_interest = ["Unnamed: 0", "title.1", "name", "bit_rate", "duration", "genre_top"]

clean = df[df['genre_top'].notnull()][columns_of_interest].copy().rename(columns={"Unnamed: 0": "fma_track_id"})

print("TOTAL CLASS DISTRIBUTION")


clean['genre_top'].value_counts() / clean.shape[0] * 100

TOTAL CLASS DISTRIBUTION


genre_top
Rock          44.797020
Electronic    33.575369
Hip-Hop       11.372531
Pop            6.334575
Classical      3.920505
Name: count, dtype: float64

In [26]:
clean['genre_top'].value_counts()

genre_top
Rock          19002
Electronic    14242
Hip-Hop        4824
Pop            2687
Classical      1663
Name: count, dtype: int64

In [20]:

utils.save_dataset(a=clean, name="tracks_clean")

# Section 2, Feature Engineering

This section needs fma_full folder unzipped under /mnt/g/ path.
The data is **bunzipped**, therefore appropriate compression/decompression algorithm must be installed. (included in linux unzip by default)

In [21]:
fma_size = "\\fma_full\\fma_full"

data_path = utils.check_os_get_root_path() + fma_size
for file in os.listdir(utils.check_os_get_artifacts_path()):
    if "tracks_clean" in file:
        tracks_clean = pd.read_parquet(f"{utils.check_os_get_artifacts_path()}/{file}").astype({'fma_track_id':'int64','name':'string', 'genre_top':'string', 'title.1': 'string'}).set_index("fma_track_id")

In next cell, we're going to grab all the folders from extracted fma_full dataset.

In [22]:
folder_paths = []

# get folders
# for folder in os.listdir(f"{data_path}/{fma_size}/"): linux
for folder in os.listdir(f"{data_path}"):
    try:
        folder_paths.append(f"{data_path}/{folder}")
    except Exception as e:
        print(f"Error occured: {e}")
        continue


df = []

for folder in folder_paths:
    folder_name = os.path.basename(folder)
    file_list = os.listdir(folder)
    id_list = [int(x.split(".")[0]) for x in file_list]

    for file_name, file_id in zip(file_list, id_list):
        df.append({
            "folder": folder_name,
            "file": file_name,
            "id": file_id,
            # "path": f"{data_path}/{folder_name}/{file_name}" LINUX
            "path": f"{data_path}\\{folder_name}\\{file_name}"
        })

df = pd.DataFrame(df)

# Important, casting the index to same type
tracks_with_paths = df.rename(columns={"id":"fma_track_id"}).set_index("fma_track_id").astype({'folder': 'string', 'file':'string', 'path':'string'})

Above's output presents all 106 thousand pieces of music mapped with their appropriate location

### Lets join the ready records and fma track's id.

In [23]:
tracks_with_paths.dtypes

folder    string[python]
file      string[python]
path      string[python]
dtype: object

In [24]:
tracks_clean.dtypes

title.1      string[python]
name         string[python]
bit_rate            float64
duration            float64
genre_top    string[python]
dtype: object

In [25]:
tracks_with_paths = tracks_clean.join(tracks_with_paths, how="left", rsuffix="tracks_id")


In [26]:
tracks_with_paths

,title.1,name,bit_rate,duration,genre_top,folder,file,path
fma_track_id,,,,,,,,
2,Food,AWOL,256000.0,168.0,Hip-Hop,000,000002.mp3,G:\\fma_full\fma_full\000\000002.mp3
3,Electric Ave,AWOL,256000.0,237.0,Hip-Hop,000,000003.mp3,G:\\fma_full\fma_full\000\000003.mp3
5,This World,AWOL,256000.0,206.0,Hip-Hop,000,000005.mp3,G:\\fma_full\fma_full\000\000005.mp3
10,Freeway,Kurt Vile,192000.0,161.0,Pop,000,000010.mp3,G:\\fma_full\fma_full\000\000010.mp3
134,Street Music,AWOL,256000.0,207.0,Hip-Hop,000,000134.mp3,G:\\fma_full\fma_full\000\000134.mp3
...,...,...,...,...,...,...,...,...
155266,I Find Myself in the Dark Woods,ps,256000.0,172.0,Electronic,155,155266.mp3,G:\\fma_full\fma_full\155\155266.mp3
155267,Take a Moment to Understand Your Internal Guid...,ps,256000.0,239.0,Electronic,155,155267.mp3,G:\\fma_full\fma_full\155\155267.mp3
155268,Air of a Certain Nervous Stillness,ps,256000.0,180.0,Electronic,155,155268.mp3,G:\\fma_full\fma_full\155\155268.mp3


# Removing Songs that are less than 15 Seconds

In [27]:
df = tracks_with_paths
df = df.reset_index()
df = df[df['duration'].ge(15)]
utils.save_dataset(df, "tracks_with_path")

# Extracting the signal from raw audio data

In [28]:
from IPython.display import clear_output

def extract_y(path: str):
    """
    takes in path of a file, loads it, and returns a ndarray of samples
    """
    # debug only
    
    track = int(path.split("\\")[-1].split(".")[0])
    print(f"{(track / 155278) * 100:.4f} %")
    print(f"track = {track}")

    clear_output(wait=True)
    
    try:
        y, _ = librosa.load(path)
        if utils.assert_at_least_minute(y, _) == True:
            try:
                y_minute = utils.get_from_middle(y, _, 15) # takes in signal, gets middle index, grabs 15//2 secs from each side
                y_minute = utils.normalize_audio(y_minute) # normalizes signal -1:1
                y_minute = utils.get_hanned(1, y_minute, _, False) # hann's the audio, so no glitches at start and at the end

                # debug only
                # print(f"y_minute.shape: {y_minute.shape}")
                # print(f"y_minute values: {y_minute}")
                # print(f"y_minute dtype: {y_minute.dtype}")

                return pd.Series([track, y_minute.tolist()])
            
            except Exception as e:
                print(f"Error during slicing a record to a minute! {track}, {e}")
                return pd.Series([track, np.nan])
        else:
            print(f"Record {path} was not long enough.")
            return pd.Series([track, np.nan])
            
        
    except Exception as e:
        print(f"Error has occured in record {path}.\n Exception: {e}\n.")
        return pd.Series([track, np.nan])

# Establishing transfer in batches to lower memory consumption

In [31]:
async def print_indexes(start, end):
    print(f"provided: {start}, {end}")
    start_index = end - (batch_size)
    print(f"Calculating for indexes: {start_index}, {end}")
    print(f"{start / len(indexes1) * 100:.2f}%") 

    # this extract y udf is the expensive part
    output = mock['path'].iloc[start_index:end].apply(extract_y)
    output.columns = ["track", "y_minute"]
    # output.to_parquet(f"/mnt/g/artifacts/{start_index}-{end}.parquet.gzip", compression="gzip") LINUX
    output.to_parquet(f"G:\\artifacts\\{start_index}-{end}.parquet.gzip", compression="gzip")


In [35]:
mock = tracks_with_paths.copy()

batch_size = 256



# indexes_custom_example = [x * batch_size for x in range(0, (mock.shape[0] // batch_size) + 1, 1)[1:]][99:]
indexes = [x * batch_size for x in range(0, (mock.shape[0] // batch_size) + 1, 1)]
indexes.insert(0, 0) # if starting from 0

# asyncio setup
tasks = [print_indexes(end, start) for end, start in enumerate(indexes)]

In [36]:
# Beware, CPU INTENSIVE TASK (batches of 512 take up to 10Gb of RAM)
# results = await asyncio.gather(*tasks)



# !!!!!!!!!!

# Finalization
After the the very intensive transformation is finished, the **artifacts** folder should be populated with batches of exported music *batch_size* in following matter:

```
['0-256.parquet.gzip',
 '1024-1280.parquet.gzip',
 '10240-10496.parquet.gzip',
 '10496-10752.parquet.gzip',
 '10752-11008.parquet.gzip',
 '11008-11264.parquet.gzip',
[etc..]]
```

In [37]:
os.listdir(utils.check_os_get_artifacts_path())

['0-256.parquet.gzip',
 '1024-1280.parquet.gzip',
 '10240-10496.parquet.gzip',
 '10496-10752.parquet.gzip',
 '10752-11008.parquet.gzip',
 '11008-11264.parquet.gzip',
 '11264-11520.parquet.gzip',
 '11520-11776.parquet.gzip',
 '11776-12032.parquet.gzip',
 '12032-12288.parquet.gzip',
 '12288-12544.parquet.gzip',
 '12544-12800.parquet.gzip',
 '1280-1536.parquet.gzip',
 '12800-13056.parquet.gzip',
 '13056-13312.parquet.gzip',
 '13312-13568.parquet.gzip',
 '13568-13824.parquet.gzip',
 '13824-14080.parquet.gzip',
 '14080-14336.parquet.gzip',
 '14336-14592.parquet.gzip',
 '14592-14848.parquet.gzip',
 '14848-15104.parquet.gzip',
 '15104-15360.parquet.gzip',
 '1536-1792.parquet.gzip',
 '15360-15616.parquet.gzip',
 '15616-15872.parquet.gzip',
 '15872-16128.parquet.gzip',
 '16128-16384.parquet.gzip',
 '16384-16640.parquet.gzip',
 '16640-16896.parquet.gzip',
 '16896-17152.parquet.gzip',
 '17152-17408.parquet.gzip',
 '17408-17664.parquet.gzip',
 '17664-17920.parquet.gzip',
 '1792-2048.parquet.gzip',

# Mapowanie sygnałów do utworów 
# (Oraz problemy z pamięcią :-) )

In [38]:
batch_size = 256
sr = 22050 # sampling rate
batches_path = utils.check_os_get_artifacts_path()

In [48]:
present = pd.read_csv("artifacts/songs_ids_with_archives.csv")
a = present.drop_duplicates(['archive'])
available = list(a['archive'])

# available

In [40]:
def read_parquet_create_path(file: str):
    """
    takes in file from os.listdir and extracts data from the dataframe, providing the track_id, fma_track_id 
    and archive name. 

    Saving each song in memory would be expensive, therefore we're storing values in Disk.
    """
    try:
        df = pd.read_parquet(f"{batches_path}{file}") # CPU intensive
        extracted = pd.DataFrame(columns=['track_id','fma_track_id','archive']).set_index("fma_track_id")
        extracted['track_id'] = df['track']
        extracted['archive'] = file

        extracted.to_csv('artifacts/songs_ids_with_archives.csv', mode='a', header=False)
        
    except Exception as e:
        print(f"exception occured on file {file}.")

In [41]:
files = os.listdir("G:\\artifacts")
not_ready = []
for file in files:
    if file not in available:
        if "tracks" not in file:
            not_ready.append(file)

not_ready[:5]

['22528-22784.parquet.gzip']

In [42]:

# CPU times: total: 53min 58s
# Wall time: 54min 22s


# CPU INTENSIVE
# %%time
# for index, file in enumerate(not_ready):
#     print(f"{index / len(files) * 100:.4f}% done")
#     read_parquet_create_path(file)

# Section 3, Class Imbalance, Hardware Limitations and Conclusions

In [43]:
import utils
df = utils.load_tracks_with_paths()
df

tracks_with_path_25-04-21-2359.parquet


,fma_track_id,title.1,name,bit_rate,duration,genre_top,folder,file,path
index,,,,,,,,,
0,2,Food,AWOL,256000.0,168.0,Hip-Hop,000,000002.mp3,G:\\fma_full\fma_full\000\000002.mp3
1,3,Electric Ave,AWOL,256000.0,237.0,Hip-Hop,000,000003.mp3,G:\\fma_full\fma_full\000\000003.mp3
2,5,This World,AWOL,256000.0,206.0,Hip-Hop,000,000005.mp3,G:\\fma_full\fma_full\000\000005.mp3
3,10,Freeway,Kurt Vile,192000.0,161.0,Pop,000,000010.mp3,G:\\fma_full\fma_full\000\000010.mp3
4,134,Street Music,AWOL,256000.0,207.0,Hip-Hop,000,000134.mp3,G:\\fma_full\fma_full\000\000134.mp3
...,...,...,...,...,...,...,...,...,...
42413,155266,I Find Myself in the Dark Woods,ps,256000.0,172.0,Electronic,155,155266.mp3,G:\\fma_full\fma_full\155\155266.mp3
42414,155267,Take a Moment to Understand Your Internal Guid...,ps,256000.0,239.0,Electronic,155,155267.mp3,G:\\fma_full\fma_full\155\155267.mp3
42415,155268,Air of a Certain Nervous Stillness,ps,256000.0,180.0,Electronic,155,155268.mp3,G:\\fma_full\fma_full\155\155268.mp3


# Class imbalance?
Niestety, ze względu na nierówny podział klas, należy podjąć decyzję odnośnie danych uczących wprowadzanych do modelu.

### Podejście 1. Redukcja

In [44]:
classical_count = df[df['genre_top'] == 'Classical'].shape[0] # 1663

1 - classical_count / df['genre_top'].value_counts()

genre_top
Rock          0.912405
Electronic    0.881825
Hip-Hop       0.650432
Pop           0.335204
Classical     0.000000
Name: count, dtype: float64

### Problem
Jeśli mielibyśmy zredukować każdy z gatunków do 1663, aby wyrównać do ilości rekordów klasycznych. 

W wyniku utracilibyśmy:
- 91% utworów Rockowych
- 88% utworów elektornicznych
- 65% utworów hip-hop
- oraz 38% utworów Popowych

# Podejście 2 - Uśrednianie

To tworzenie nowych przykładów danych przez średnie ważone lub arytmetyczne połączenie dwóch (lub więcej) istniejących przykładów. W kontekście klasyfikacji może to wyglądać tak:

new_sample = 0.5 * piosenka1 + 0.5 * piosenka2

lub

new_sample = 0.7 * sample1 + 0.3 * sample2


Samo uśrednianie może zaburzyć strukturę utworu, i nadaje się bardziej do większych ilości danych. Niestety, nie tworzy syntetycznych utworów, jedynie je uśrednia.

# Podejście 3 - GAN
https://developers.google.com/machine-learning/gan?hl=pl

Ze względu na obliczeniowy koszt związany ze szkoleniem sieci neuronowych, nie jest możliwe stworzenie satysfakcjonującego modelu Głębokiej Splotowej Generatywnej Przeciwstawnej Sieci Neuronowej.

Samo utrzymywanie 200 utworów o długości 1 minuty zajmuje 10Gb pamięci RAM, nie mówiąc o szkoleniu na karcie Geforce RTX 3060Ti, posiadającej 12Gb VRAM. Z tego powodu, podjęta została decyzja o zmniejszeniu ilości przykładów uczenia do 500(?) per gatunek, a ich długość została ograniczona do 15 sekund.

Wraz z zastosowaniem obliczeń w chmurze, wykorzystaniu uczenia rozproszonego w klastrze lub 
zastosowania większej, silniejszej maszyny, moglibyśmy osiągnąć potężne rozwiązanie do generowania muzyki.

In [7]:
import utils
utils.import_data_science(globals())
artifacts_path = utils.check_os_get_artifacts_path()

signals = pd.read_csv("artifacts/songs_ids_with_archives.csv")

signals = signals.astype({
    'fma_track_id': 'int64',
    'track_id': 'int64',
    'archive': 'string'
}).set_index("fma_track_id")

signals

,track_id,archive
fma_track_id,,
2,2,0-256.parquet.gzip
3,3,0-256.parquet.gzip
5,5,0-256.parquet.gzip
10,10,0-256.parquet.gzip
134,134,0-256.parquet.gzip
...,...,...
45372,45372,9984-10240.parquet.gzip
45373,45373,9984-10240.parquet.gzip
45374,45374,9984-10240.parquet.gzip


In [8]:
def extract_f32_from_str(row: str) -> np.array:
    try:
        no_brackets = row[1:-1]
        splitted    = no_brackets.split(",")
        casted      = np.array(splitted, dtype=np.float32)
        
        return casted
    except Exception as e:
        # print(f"lost in conversion. {e}")
        return np.array(np.nan)

In [9]:
songs = pd.read_parquet("datasets/records_ready.parquet")

songs = songs.astype({"fma_track_id": "int64",}).set_index("fma_track_id")


In [10]:
master = signals.join(songs).dropna(subset=['archive'])

In [12]:
master_rock = master[master['genre_top'] == "Rock"][:1400]
master_hh = master[master['genre_top'] == "Hip-Hop"][:1400]
master_ele = master[master['genre_top'] == "Electronic"][:1400]
master_pop = master[master['genre_top'] == "Pop"][:1400]
master_class = master[master['genre_top'] == "Classical"][:1400]

In [13]:
master = pd.concat([master_rock, master_hh, master_ele, master_pop, master_class])
master = master[['genre_top','archive']]

In [14]:
master['genre_top'].value_counts()

genre_top
Rock          1400
Hip-Hop       1400
Electronic    1400
Pop           1400
Classical     1400
Name: count, dtype: int64

In [15]:
archives = list(master['archive'].unique())
len(archives)

124

In [16]:
master['y_minute'] = np.nan
master = master.reset_index()

In [17]:
master = np.array(master)
master_index = master[:, 0]

In [18]:
master

array([[135, 'Rock', '0-256.parquet.gzip', nan],
       [136, 'Rock', '0-256.parquet.gzip', nan],
       [151, 'Rock', '0-256.parquet.gzip', nan],
       ...,
       [36537, 'Classical', '8448-8704.parquet.gzip', nan],
       [36538, 'Classical', '8448-8704.parquet.gzip', nan],
       [36539, 'Classical', '8448-8704.parquet.gzip', nan]],
      shape=(7000, 4), dtype=object)

In [19]:
from IPython.display import clear_output
import pyarrow.parquet as pq

for archive in archives:
    table = np.array(pq.read_table(f"{artifacts_path}{archive}"))
    
    for track_id in table:
    
        try:
            track_location = int(np.where(master[:,0] == track_id[0])[0][0])
            # print("index_found")
            # print(f"fma_track_id from archive with signal: {track_id[0]}")
            # print(f"track location in master: {track_location}")
            master[track_location][3] = extract_f32_from_str(track_id[2])
            arr = master[:, 3]
            is_nan_mask = np.array([isinstance(x, float) and np.isnan(x) for x in arr])
            print(f"currently {np.sum(~is_nan_mask) / master.shape[0] * 100}")
            clear_output(wait=True)
            
        except:
            pass
np.save('/mnt/g/artifacts/signals7k.npy', master)

currently 100.0


In [ ]:
np.save('artifacts/signals.npy', master)

# NOTHING ELSE AFTER THIS POINT IS NEEDED

In [ ]:
## append version Bro pandas are way too slow xD
# import gc

# for archive in archives:
#     signal_df = pd.read_parquet(f"{artifacts_path}{archive}")
#     master.update(signal_df['y_minute'])
#     current_with_signal = master[master['y_minute'].notnull()]
#     print(f"shape current_with_signal: {current_with_signal.shape}")
#     current_with_signal.to_json(lines=True,
#                                 orient='records',
#                                 mode='a',
#                                 path_or_buf='/mnt/g/artifacts/master_train_append.json')
#     current_indexes = current_with_signal.index
#     master.drop(current_indexes, inplace=True)
    
#     del current_indexes
#     del current_with_signal
#     del signal_df
    
    

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



shape current_with_signal: (163, 3)
current_indexes: Index([135, 136, 151, 152, 153, 154, 155, 169, 170, 171,
       ...
       658, 659, 660, 661, 662, 663, 664, 665,  10, 213],
      dtype='int64', name='fma_track_id', length=163)
current master:               genre_top                   archive y_minute
fma_track_id                                              
318                Rock        0-256.parquet.gzip      NaN
342                Rock        0-256.parquet.gzip      NaN
46078           Hip-Hop  10240-10496.parquet.gzip      NaN
46726           Hip-Hop  10240-10496.parquet.gzip      NaN
46727           Hip-Hop  10240-10496.parquet.gzip      NaN
...                 ...                       ...      ...
50940         Classical  11008-11264.parquet.gzip      NaN
50941         Classical  11008-11264.parquet.gzip      NaN
50942         Classical  11264-11520.parquet.gzip      NaN
50943         Classical  11264-11520.parquet.gzip      NaN
50944         Classical  11264-11520.parque

In [68]:
# for archive in archives:
#     signal_df = pd.read_parquet(f"{artifacts_path}{archive}")
#     master.update(signal_df['y_minute'])
#     del signal_df
#     print(master['y_minute'].notnull().value_counts())


# print("finished")
# # 500 songs of each genre consumes 25Gb's of RAM
# master

In [14]:
# utils.save_dataset(a=master, name="train_ready")


# master = master[master['y_minute'].notnull()].rename(columns={'genre_top': 'genre'})


In [15]:
# master['genre'].value_counts()

genre
Hip-Hop       50
Electronic    50
Rock          49
Classical     49
Pop           47
Name: count, dtype: int64

In [16]:
# master.memory_usage(deep=True) // 1_000_000

Index          0
track_id       0
archive        0
title.1        0
name           0
bit_rate       0
duration       0
genre          0
y_minute    1552
dtype: int64

In [17]:
# master.to_json("G:\\artifacts\\master_train.json")


In [25]:
# import utils

# train_data.to_json(utils.check_os_get_artifacts_path() + "master_train.json")
    